In [ ]:
"""
Two tiny worked examples of Groebner basis computations, illustrating the
two outcomes that showed up in the Sigma'/Gamma' search:
  - basis == {1}                -> no solution exists at all
  - basis == {the raw unknowns} -> only the trivial solution exists

The general recipe, same in both cases:
  1. Write the invariance condition "if y = (line), then ydot = m*xdot"
     (chain rule for a curve to stay on itself).
  2. Substitute the ansatz, expand, and collect coefficients of each power
     of x -- every coefficient must vanish for the line to be invariant
     for ALL x, not just one point.
  3. Feed that list of coefficient-equations (now polynomials in the
     unknowns m, b) into sp.groebner() and read off what it implies.
"""
import sympy as sp

x, y, m, b = sp.symbols('x y m b', real=True)

print("=== Example 1: xdot = y, ydot = x^2 ===")
print("Looking for an invariant line y = m*x + b\n")

xdot, ydot = y, x**2
y_line = m*x + b

# invariance: ydot must equal m*xdot, evaluated ON the line
residual = sp.expand(ydot.subs(y, y_line) - m*xdot.subs(y, y_line))
coeffs = sp.Poly(residual, x).coeffs()

print(f"residual (must vanish for every x): {residual}")
print(f"coefficients of x^2, x^1, x^0:      {coeffs}")

G1 = sp.groebner(coeffs, m, b)
print(f"Groebner basis: {list(G1.polys)}")
print(f"Is 1 in the basis? {1 in G1}  ->  {'NO invariant line exists at all' if 1 in G1 else 'solutions exist'}\n")


print("=== Example 2: xdot = x, ydot = 2y ===")
print("Looking for an invariant line through the origin, y = m*x\n")

xdot2, ydot2 = x, 2*y
y_line2 = m*x

residual2 = sp.expand(ydot2.subs(y, y_line2) - m*xdot2.subs(y, y_line2))
coeffs2 = sp.Poly(residual2, x).coeffs()

print(f"residual (must vanish for every x): {residual2}")
print(f"coefficient of x^1:                 {coeffs2}")

G2 = sp.groebner(coeffs2, m)
print(f"Groebner basis: {list(G2.polys)}")
print("-> forces m = 0, i.e. only the trivial line y=0 (the x-axis) is invariant,")
print("   which makes sense directly: ydot=2y=0 automatically whenever y=0.")

In [ ]:
"""
Search for affine analogs of Sigma and Gamma when kappa_a != 0.

Sigma = {A=0, P_B=0}      (kappa_a=0 invariant plane containing S, M_+, M_-)
Gamma = {P_A=A0-A, P_B=-B} (kappa_a=0 invariant plane containing W_+, M_+)

For kappa_a != 0, neither survives in its original form. Here we search for
a generalized AFFINE surface of the same type ("two coordinates as affine
functions of the other two") and use a Groebner basis to determine, once
and for all, whether such a surface exists -- not just whether sympy's
solve() happens to find one.
"""
import sympy as sp

A, B, PA, PB = sp.symbols('A B P_A P_B', real=True)
g, A_0_p_sq, ka, kb = sp.symbols('g varepsilon_d kappa_a kappa_b', positive=True)
sqrt2 = sp.sqrt(2)

# full equations of motion (kappa_a != 0)
dA  = sqrt2*g*(PA*PB - A*B) - ka*PA
dB  = g/sqrt2*(A**2+PA**2-A_0_p_sq) - kb/2*B - kb*PB
dPA = sqrt2*g*(B*PA - A*PB) 
dPB = sqrt2*g*A*PA + kb/2*PB


def search_affine_invariant(dep1, dep2, indep1, indep2, dot):
    """
    Look for an affine invariant surface where dep1, dep2 are affine
    functions of indep1, indep2:
        dep1 = c0 + c1*indep1 + c2*indep2
        dep2 = d0 + d1*indep1 + d2*indep2

    dot: dict mapping each of the four symbols to its equation of motion
         (dA, dB, dPA, dPB as appropriate).

    Returns the Groebner basis of the coefficient ideal in (c0,c1,c2,d0,d1,d2).
    """
    c0, c1, c2, d0, d1, d2 = sp.symbols('c0 c1 c2 d0 d1 d2')
    dep1_ansatz = c0 + c1*indep1 + c2*indep2
    dep2_ansatz = d0 + d1*indep1 + d2*indep2
    subs_map = {dep1: dep1_ansatz, dep2: dep2_ansatz}

    dot_dep1_sub  = sp.expand(dot[dep1].subs(subs_map))
    dot_dep2_sub  = sp.expand(dot[dep2].subs(subs_map))
    dot_indep1_sub = sp.expand(dot[indep1].subs(subs_map))
    dot_indep2_sub = sp.expand(dot[indep2].subs(subs_map))

    # invariance: d/dt(dep1) must equal c1*indep1dot + c2*indep2dot
    #             d/dt(dep2) must equal d1*indep1dot + d2*indep2dot
    eq1 = sp.expand(dot_dep1_sub - c1*dot_indep1_sub - c2*dot_indep2_sub)
    eq2 = sp.expand(dot_dep2_sub - d1*dot_indep1_sub - d2*dot_indep2_sub)

    coeffs = (sp.Poly(eq1, indep1, indep2).coeffs()
              + sp.Poly(eq2, indep1, indep2).coeffs())

    return sp.groebner(coeffs, c0, c1, c2, d0, d1, d2, order='lex'), (c0, c1, c2, d0, d1, d2)


dot = {A: dA, B: dB, PA: dPA, PB: dPB}

print("=== Searching for Sigma': A, P_B affine functions of (B, P_A) ===")
G_sigma, unknowns_sigma = search_affine_invariant(A, PB, B, PA, dot)
print("Groebner basis:", [p.as_expr() for p in G_sigma.polys])
print("Is 1 in the basis (i.e. NO solution at all)?", 1 in G_sigma)

print("\n=== Searching for Gamma': P_A, P_B affine functions of (A, B) ===")
G_gamma, unknowns_gamma = search_affine_invariant(PA, PB, A, B, dot)
print("Groebner basis:", [p.as_expr() for p in G_gamma.polys])
print("Basis equals the raw unknowns (i.e. ONLY the trivial p=q=0 solution, = Pi)?",
      all(sp.Poly(p, *unknowns_gamma).is_monomial and sp.total_degree(p.as_expr()) == 1
          for p in G_gamma.polys) and len(G_gamma.polys) == len(unknowns_gamma))

print("\n=== Sanity check: at kappa_a=0, the ORIGINAL Gamma should solve the Gamma' equations ===")
#A0 = sp.sqrt(eps_d/g)
#old_gamma = {unknowns_gamma[0]: A0, unknowns_gamma[1]: -1, unknowns_gamma[2]: 0,
#             unknowns_gamma[3]: 0, unknowns_gamma[4]: 0, unknowns_gamma[5]: -1}
# rebuild the eq1/eq2 residuals at ka=0 and check they vanish under old_gamma
#c0, c1, c2, d0, d1, d2 = unknowns_gamma
#dep1_ansatz = c0 + c1*A + c2*B
#dep2_ansatz = d0 + d1*A + d2*B
#eq1 = sp.expand(dPA.subs({PA: dep1_ansatz, PB: dep2_ansatz}) - c1*dA.subs({PA: dep1_ansatz, PB: dep2_ansatz}) - c2*dB.subs({PA: dep1_ansatz, PB: dep2_ansatz}))
#residual_check = sp.simplify(eq1.subs(old_gamma).subs(ka, 0))
#print("residual of old Gamma plugged into the Gamma' equations, at ka=0:", residual_check, " (should be 0)")